In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import KNNImputer
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from functions.functions import *

In [ ]:
df = pd.read_csv('../data/source/df_joined.csv')
df['default'] = df['default'].astype('int')
df['obs_date'] = pd.to_datetime(df['obs_date'])
df['year'] = df['obs_date'].dt.year

In [ ]:
df

In [ ]:
# do we need this? can we delete it?
df_slice = df[df['ID']==1574]
df_slice2 = df[df['ID']==9353]

In [ ]:
df_1 = filter_df(df)
# df_1 = df_1[['ID', 'obs_date', 'cutoff_date', 'default', 'new_default']]

In [ ]:
df_1

## Check default ratio in every year

In [ ]:
df_1.groupby('year')['default'].mean()

In [ ]:
var_columns = [f'Var_{i:02d}' for i in range(1, 40)]

In [ ]:
df_1 = preprocess_data(df_1)

In [ ]:
df_1

## Fill missing values with KNN imputer

In [ ]:
imputer = KNNImputer(n_neighbors=5)

In [ ]:
df_imputed = pd.DataFrame(imputer.fit_transform(df_1[var_columns]), columns=var_columns)

In [ ]:
default_column = df_1['default'].reset_index(drop=True)

# Assign the reset index column to df_imputed
df_imputed['default'] = default_column

## Weight of evidence

In [ ]:
df_binned = binning_with_decision_tree(df_imputed)

In [ ]:
df_woe = calculate_woe(df_binned, target='default')

In [ ]:
df_woe

In [ ]:
# df_woe.to_csv('../data/processed/df_woe.csv', index=False)

In [ ]:
# df_woe = pd.read_csv('../data/processed/df_woe.csv')

## Population stability index (PSI)

In [ ]:
# Preparation for PSI calculation in order to check the colinearity. 
df_woe_time = df_woe
df_woe_time['year'] = df_1['year'].reset_index(drop=True)

In [ ]:
psi_results = calculate_psi(df_woe_time, time_column='year')

In [ ]:
psi_results.head(5)

## Check colinearity

In [ ]:
df_VIF_woe = check_collinearity(df_woe)

In [ ]:
param_grid = {
        'penalty': ['l1', 'l2'],  # Regularization penalty
        'C': [0.001, 0.01, 0.1, 1],  # Inverse of regularization strength
        'solver': ['liblinear', 'saga'],  # Algorithm to use in the optimization problem
        'max_iter': [100, 500, 1000],  # Maximum number of iterations
        'class_weight': [None, 'balanced']  # Weights associated with classes
    }

model = LogisticRegression(random_state = 42)

In [ ]:
# columns to drop after VIW calculation
cols_to_drop = ['Var_02_woe', 'Var_01_woe', 'Var_11_woe', 'Var_18_woe', 'Var_29_woe', 'Var_36_woe', 'Var_34_woe', 'Var_22_woe', 'Var_12_woe', 'Var_16_woe', 'Var_19_woe', 'Var_30_woe', 'Var_09_woe', 'Var_21_woe', 'Var_17_woe', 'Var_14_woe', 'Var_26_woe', 'Var_32_woe', 'Var_39_woe', 'Var_24_woe', 'Var_06_woe', 'Var_38_woe', 'Var_10_woe', 'Var_20_woe']
df_VIF_woe = df_woe.drop(cols_to_drop, axis=1)

In [ ]:
# df_VIF_woe.to_csv('../data/processed/df_VIF_woe.csv', index=False)

## Estimate Logistic regresison

In [ ]:
results_vif_LR = train_and_evaluate_pipeline(df_VIF_woe, time = 2020, model = model, param_grid = param_grid)

## Estimate XGB

In [ ]:
from xgboost import XGBClassifier

# Define XGBoost model
xgb_model = XGBClassifier(random_state=42)
param_grid_xgboost = {
    'learning_rate': [0.1, 0.2],
    'n_estimators': [200],
    #'max_depth': [3, 4, 5],
    'min_child_weight': [2, 3],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7],
    'gamma': [0.1, 0.2],
    'reg_alpha': [0.1, 0.2],
    #'reg_lambda': [0, 0.1, 0.2]
}


In [ ]:
results_vif_xgb = train_and_evaluate_pipeline(df_VIF_woe, target='default', time=2020, model=xgb_model, param_grid=param_grid_xgboost)

In [ ]:
results_vif_xgb

In [ ]:
results_vif_LR

### Write results of LR and XGB

In [ ]:
results_vif_LR.to_csv('../data/results/results_vif_LR.csv', index=False)
results_vif_xgb.to_csv('../data/results/results_vif_xgb.csv', index=False)

In [ ]:
# select just first row from results_vif_LR as dataframe
one_xgb = results_vif_xgb.iloc[0]
one_xgb = pd.DataFrame(one_xgb).T
one_xgb


In [ ]:
results_final = pd.concat([results_vif_LR, one_xgb])

In [ ]:
plt.figure(figsize=(10, 6))

# In-time testing Gini
plt.scatter(results_final['In_Time_Train_Gini'], results_final['In_Time_Test_Gini'], color='#000066', label='Out-of-sample')

# Out-of-time testing Gini
plt.scatter(results_final['In_Time_Train_Gini'], results_final['Out_of_Time_Test_Gini'], color='#ff6200', label='Out-of-time')

# Add lollipop lines
for index, row in results_final.iterrows():
    plt.plot([row['In_Time_Train_Gini'], row['In_Time_Train_Gini']], [row['In_Time_Test_Gini'], row['Out_of_Time_Test_Gini']], color='gray', linestyle='--', linewidth=0.8)
    plt.text(row['In_Time_Train_Gini'] - 0.002, row['In_Time_Test_Gini'] + 0.002, row['Model_Name'], ha='right', va='bottom', fontsize=8)

# Add labels and legend
plt.xlabel('Training Gini', fontsize=13)
plt.ylabel('Testing Gini', fontsize=13)
plt.title('Gini coefficient', fontsize=14, fontweight='bold')
plt.legend()

# Set limits for x and y axes
plt.xlim(0.7, 0.95)
plt.ylim(0.7, 0.95)

# Show plot
plt.grid(True)
plt.tight_layout()
plt.show()

## Shapley value

In [ ]:
df_VIF_woe = pd.read_csv('../data/processed/df_VIF_woe.csv')
results_vif_LR = pd.read_csv('../data/results/results_vif_LR.csv')
results_vif_xgb = pd.read_csv('../data/results/results_vif_xgb.csv')

In [ ]:
var_dict = {'Var_03_woe': 'Cash And Cash Equivalents', 
            'Var_08_woe': 'Earnings Before Interest And Tax', 
            'Var_25_woe': 'Quick Ratio', 
            'Var_04_woe': 'Current Ratio',
            'Var_31_woe': 'Sales / Total Assets', 
            'Var_15_woe': 'Gross Profit Margin', 
            'Var_05_woe': 'Debt / Net Worth', 
            'Var_28_woe': 'Return On Total Equity Reserve',
            'Var_27_woe': 'Return On Assets', 
            'Var_37_woe': 'Total Net Debt EBITDA', 
            'Var_13_woe': 'Financial Leverage', 
            'Var_35_woe': 'Total Liabilities / Total Assets',
            'Var_33_woe': 'Senior Net Debt', 
            'Var_23_woe': 'Net Profit Margin', 
            'Var_07_woe': 'EBITDA Margin'
            }

# change colnames in df_VIF_woe according to var_dict
df_VIF_woe.rename(columns=var_dict, inplace=True)

In [ ]:
df_VIF_woe

## Delete observations after first iteration

In [ ]:
cols_to_drop = ['Net Profit Margin', 'Return On Total Equity Reserve']
df_VIF_woe_dropped = df_VIF_woe.drop(cols_to_drop, axis=1)

In [ ]:
# df_VIF_woe_dropped.to_csv('../data/processed/df_VIF_woe_dropped.csv', index=False)

In [ ]:
df_VIF_woe_dropped = pd.read_csv('../data/processed/df_VIF_woe_dropped.csv')

In [ ]:
model_config = results_vif_LR.iloc[0,0]
model_to_shapley(df_VIF_woe_dropped, model_config)

In [ ]:
model_config